# BR09 — Initial and adjudicated clinical results

Run after every initial rating and every triggered adjudication in BR08. CPU only. Missing adjudications stop analysis; an explicitly unresolved decision is allowed but never converted to a correct report. Compare graph support against the bidirectional-pool control within patient and report cohorts/models separately.

The primary clinical endpoint is a report containing at least one clinically significant error. A negative graph-minus-pool difference favors graph. Confidence intervals reflect case sampling, not differences between clinicians. There is no inter-rater reliability estimate with one clinician.

> **Clinical review:** original helpers are included in `clinical_review/`. Completed reviewer data are not distributed. See `docs/RUN_ORDER.md` and `docs/REPRODUCIBILITY.md`.

In [ ]:
from pathlib import Path
import sys
EXP = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'config.example.json').is_file() and (p / 'common.py').is_file()), None)
if EXP is None:
    raise FileNotFoundError('Start Jupyter inside graph_reasoning/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read
# Use the EXACT completed BR04B child config. Change this for later cohorts.
CONFIG_PATH = Path('/path/to/repository/outputs/real_run_001/'
                   'experiment02_v1/context_fit/v1/internal_val/config.json')
if not CONFIG_PATH.is_file():
    raise FileNotFoundError('Set CONFIG_PATH to the actual BR04B context-fit child config: ' + str(CONFIG_PATH))
CFG = load_config(CONFIG_PATH)
parent_setup(CFG)
RUN = Path(CFG['run_dir'])
REVIEW_VERSION = 'calibration_v1'  # Same value in BR07, BR08 and BR09. New version for main review.
REVIEW_DIR = RUN / 'clinical_review' / CFG['cohort'] / REVIEW_VERSION
from clinical_review.workflow import progress
print('Cohort:', CFG['cohort'], '| review:', REVIEW_DIR)
if CFG['cohort'] == 'internal_val':
    print('Validation review is calibration/exploratory, not held-out clinical validation.')


In [ ]:
from clinical_review.workflow import analyze
print(progress(REVIEW_DIR))
ANALYSIS_VERSION = 'analysis_v1'
REPEATS = 2000
RESULTS_DIR = REVIEW_DIR / ANALYSIS_VERSION
result = analyze(REVIEW_DIR, RESULTS_DIR, repeats=REPEATS, seed=CFG['seed'])
print('Complete:', result['complete'], '| cohort:', result['cohort'], '| stage:', result['stage'])
print('Adjudication decisions:', result['adjudication_decisions'])
for row in result['comparisons']:
    if row['metric'] == 'major_error': print(row)
print('Outputs:', RESULTS_DIR)


In [ ]:
# Descriptive quality summaries; inspect exclusions and generation coverage before interpretation.
for row in result['summary']: print(row)
print('Generation coverage:', result['coverage'])
for limitation in result['limitations']: print('-', limitation)
print('CSV files: per_report.csv, summary.csv, paired_comparisons.csv')
print('Narrative: findings.txt | Full results/provenance: results.json, protocol.json')


## Interpretation before manuscript drafting

Keep initial and adjudicated estimates together. Report number of unique patient IDs, sampled/evaluable reports, generation failures, adjudication decisions, unresolved and uninterpretable assessments. Lower major-error/omission rates favor graph; higher quality/acceptability favors graph. Intervals spanning zero do not establish a benefit. A sampled, single-reviewer study does not demonstrate patient outcomes or broad clinician agreement.

Run separately for each completed main cohort and preselected model subset using its exact BR04B child config. Calibration/internal-validation ratings are exploratory. Do not pool them with main outcomes. Summarize external validation separately, including label-mapping and patient-linkage limitations. The clinical endpoint complements rather than replaces the original BR05 automated endpoint. See clinical_evaluation_plan.md for manuscript deliverables.